# 04 Labeling notebook 


## Load Data

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

DATA_PATH = Path("../data/processed/online_retail_ll_features.csv")
OUTPUT_DIR = Path("../data/processed")
features=pd.read_csv(DATA_PATH)

features = features.sort_values(
    ["CustomerID", "window_id"]
).reset_index(drop=True)

features.head()

,CustomerID,window_id,window_start,window_end,orders,spend,totalQuantity,unique_products,active_days,line_items,...,prev_unique_products,prev_active_days,prev_items_per_order,orders_change_pct,spend_change_pct,totalQuantity_change_pct,avargeOrderValue_change_pct,unique_products_change_pct,active_days_change_pct,items_per_order_change_pct
0,12346,6,2010-05-30,2010-06-29,1,142.31,19,19,1,19,...,5.0,1.0,5.0,0.0,4.260998,2.800000,4.260998,2.800000,0.0,2.800000
1,12346,13,2010-12-26,2011-01-25,1,77183.60,74215,1,1,1,...,19.0,1.0,19.0,0.0,541.362448,3905.052632,541.362448,-0.947368,0.0,-0.947368
2,12347,12,2010-11-26,2010-12-26,1,711.79,319,31,1,31,...,40.0,1.0,40.0,0.0,0.163949,-0.373281,0.163949,-0.225000,0.0,-0.225000
3,12347,14,2011-01-25,2011-02-24,1,475.39,315,29,1,29,...,31.0,1.0,31.0,0.0,-0.332120,-0.012539,-0.332120,-0.064516,0.0,-0.064516
4,12347,16,2011-03-26,2011-04-25,1,636.25,483,24,1,24,...,29.0,1.0,29.0,0.0,0.338375,0.533333,0.338375,-0.172414,0.0,-0.172414


## Create Future Values to prevent Target leakge 
 

In [2]:
features["next_orders"] = (
    features.groupby("CustomerID")["orders"].shift(-1)
)

features["next_spend"] = (
    features.groupby("CustomerID")["spend"].shift(-1)
)
features.head()

,CustomerID,window_id,window_start,window_end,orders,spend,totalQuantity,unique_products,active_days,line_items,...,prev_items_per_order,orders_change_pct,spend_change_pct,totalQuantity_change_pct,avargeOrderValue_change_pct,unique_products_change_pct,active_days_change_pct,items_per_order_change_pct,next_orders,next_spend
0,12346,6,2010-05-30,2010-06-29,1,142.31,19,19,1,19,...,5.0,0.0,4.260998,2.800000,4.260998,2.800000,0.0,2.800000,1.0,77183.60
1,12346,13,2010-12-26,2011-01-25,1,77183.60,74215,1,1,1,...,19.0,0.0,541.362448,3905.052632,541.362448,-0.947368,0.0,-0.947368,NaN,NaN
2,12347,12,2010-11-26,2010-12-26,1,711.79,319,31,1,31,...,40.0,0.0,0.163949,-0.373281,0.163949,-0.225000,0.0,-0.225000,1.0,475.39
3,12347,14,2011-01-25,2011-02-24,1,475.39,315,29,1,29,...,31.0,0.0,-0.332120,-0.012539,-0.332120,-0.064516,0.0,-0.064516,1.0,636.25
4,12347,16,2011-03-26,2011-04-25,1,636.25,483,24,1,24,...,29.0,0.0,0.338375,0.533333,0.338375,-0.172414,0.0,-0.172414,1.0,382.52


## Measure Future Order and Spend Changes

These values are calculated to measure changes in a customer's future purchasing behavior. They are used to determine whether the customer exhibits a **Behavior Shift**, based on a 50% decrease threshold in either order count or total spend.


In [3]:
features["future_orders_change"] = (
    (features["next_orders"] - features["orders"])
    / features["orders"]
)

features["future_spend_change"] = (
    (features["next_spend"] - features["spend"])
    / features["spend"]
)

## Measure  User Behavior Shift
Threshold : 30%

In [4]:
features["BehaviorShift"] = (
    (features["future_orders_change"] <= -0.30)
    |
    (features["future_spend_change"] <= -0.30)
).astype(int)

In [5]:
features.head()

,CustomerID,window_id,window_start,window_end,orders,spend,totalQuantity,unique_products,active_days,line_items,...,totalQuantity_change_pct,avargeOrderValue_change_pct,unique_products_change_pct,active_days_change_pct,items_per_order_change_pct,next_orders,next_spend,future_orders_change,future_spend_change,BehaviorShift
0,12346,6,2010-05-30,2010-06-29,1,142.31,19,19,1,19,...,2.800000,4.260998,2.800000,0.0,2.800000,1.0,77183.60,0.0,541.362448,0
1,12346,13,2010-12-26,2011-01-25,1,77183.60,74215,1,1,1,...,3905.052632,541.362448,-0.947368,0.0,-0.947368,NaN,NaN,NaN,NaN,0
2,12347,12,2010-11-26,2010-12-26,1,711.79,319,31,1,31,...,-0.373281,0.163949,-0.225000,0.0,-0.225000,1.0,475.39,0.0,-0.332120,1
3,12347,14,2011-01-25,2011-02-24,1,475.39,315,29,1,29,...,-0.012539,-0.332120,-0.064516,0.0,-0.064516,1.0,636.25,0.0,0.338375,0
4,12347,16,2011-03-26,2011-04-25,1,636.25,483,24,1,24,...,0.533333,0.338375,-0.172414,0.0,-0.172414,1.0,382.52,0.0,-0.398790,1


In [6]:
labeled = features.dropna(subset=["future_orders_change", "future_spend_change"])

users_with_shift = labeled.loc[
    labeled["BehaviorShift"] == 1, "CustomerID"
].nunique()

print("Users with at least one behavior shift:", users_with_shift)
print(labeled["BehaviorShift"].value_counts())

Users with at least one behavior shift: 2235
BehaviorShift
0    10088
1     5494
Name: count, dtype: int64


In [7]:
features["BehaviorShift"].value_counts(normalize=True)

BehaviorShift
0    0.72089
1    0.27911
Name: proportion, dtype: float64

In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUTPUT_DIR / f"online_retail_II_labeled_30.csv"
labeled.to_csv(out_path, index=False)
print(f"Saved: {out_path.resolve()} | shape={labeled.shape}")

Saved: C:\Users\HP\Desktop\Projects\customer-behavior-shift-detection\data\processed\online_retail_II_labeled_30.csv | shape=(15582, 34)
